In [ ]:
# In[1]:

import pandas as pd
import pytz

# timezone for UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Load CSVs
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_err = pd.read_csv('error_logs.csv')

# Ensure timestamp numeric
for df in (df_metric, df_trace, df_log, df_err):
    df['timestamp'] = pd.to_numeric(df['timestamp'], errors='coerce').astype('Int64')

# --- Metrics aggregation ---
# basic aggregations
metric_grp = df_metric.groupby(['cmdb_id', 'kpi_name'], dropna=False).agg(
    count=('value', 'count'),
    min_timestamp=('timestamp', 'min'),
    max_timestamp=('timestamp', 'max')
).reset_index()

# compute global 95th percentile per group (using full series)
metric_p95 = df_metric.groupby(['cmdb_id', 'kpi_name'], dropna=False)['value'].quantile(0.95).rename('global_P95_value').reset_index()

# merge
metric_agg = metric_grp.merge(metric_p95, on=['cmdb_id', 'kpi_name'], how='left')

# sort by count desc and keep top 20
metric_agg = metric_agg.sort_values('count', ascending=False).reset_index(drop=True)
metric_agg_head = metric_agg.head(20)

# --- Trace aggregation ---
trace_agg = df_trace.groupby(['cmdb_id', 'trace_name'], dropna=False).agg(
    count=('value', 'count'),
    min_timestamp=('timestamp', 'min'),
    max_timestamp=('timestamp', 'max')
).reset_index().sort_values('count', ascending=False).reset_index(drop=True)
trace_agg_head = trace_agg.head(20)

# --- Log aggregation ---
log_agg = df_log.groupby(['cmdb_id', 'log_name'], dropna=False).agg(
    count=('value', 'count'),
    min_timestamp=('timestamp', 'min'),
    max_timestamp=('timestamp', 'max')
).reset_index().sort_values('count', ascending=False).reset_index(drop=True)
log_agg_head = log_agg.head(20)

# --- Error logs summary ---
error_count = int(df_err.shape[0])
error_sample = df_err[['timestamp', 'cmdb_id', 'message']].head(20)

# --- Overall min/max timestamps per file (also converted to Asia/Shanghai) ---
def mk_range(df, name):
    if df['timestamp'].dropna().shape[0] == 0:
        return {'file': name, 'min_ts': pd.NA, 'max_ts': pd.NA, 'min_dt_local': pd.NA, 'max_dt_local': pd.NA}
    min_ts = int(df['timestamp'].min())
    max_ts = int(df['timestamp'].max())
    min_dt_local = pd.to_datetime(min_ts, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
    max_dt_local = pd.to_datetime(max_ts, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
    return {'file': name, 'min_ts': min_ts, 'max_ts': max_ts, 'min_dt_local': min_dt_local, 'max_dt_local': max_dt_local}

ranges = pd.DataFrame([
    mk_range(df_metric, 'metric.csv'),
    mk_range(df_trace, 'trace.csv'),
    mk_range(df_log, 'log.csv'),
    mk_range(df_err, 'error_logs.csv'),
])

# Keep outputs compact: show top-20 aggregated tables and concise summaries
metric_agg_head, trace_agg_head, log_agg_head, error_count, error_sample, ranges

```
Out[1]:
```
# concise plain-English summary based on prior aggregations stored in kernel variables
# Reuse existing variables: metric_agg_head, trace_agg_head, log_agg_head, error_count, error_sample, ranges

lines = []

# Metrics summary
if 'metric_agg_head' in globals() and not metric_agg_head.empty:
    lines.append(f"Metrics: showing top {min(20, len(metric_agg_head))} (by count).")
    # include top 3 entries
    top_n = metric_agg_head.head(3)
    for i, row in top_n.iterrows():
        lines.append(f"  {i+1}. {row['cmdb_id']} - {row['kpi_name']} | count={int(row['count'])}, "
                     f"min_ts={int(row['min_timestamp'])}, max_ts={int(row['max_timestamp'])}, "
                     f"P95={float(row['global_P95_value']):.4g}")
else:
    lines.append("Metrics: no metric data found.")

# Trace summary
if 'trace_agg_head' in globals() and not trace_agg_head.empty:
    lines.append(f"Traces: showing top {min(20, len(trace_agg_head))} (by count).")
    top_n = trace_agg_head.head(3)
    for i, row in top_n.iterrows():
        lines.append(f"  {i+1}. {row['cmdb_id']} - {row['trace_name']} | count={int(row['count'])}, "
                     f"min_ts={int(row['min_timestamp'])}, max_ts={int(row['max_timestamp'])}")
else:
    lines.append("Traces: no trace data found.")

# Logs summary
if 'log_agg_head' in globals() and not log_agg_head.empty:
    lines.append(f"Logs: showing top {min(20, len(log_agg_head))} (by count).")
    top_n = log_agg_head.head(3)
    for i, row in top_n.iterrows():
        lines.append(f"  {i+1}. {row['cmdb_id']} - {row['log_name']} | count={int(row['count'])}, "
                     f"min_ts={int(row['min_timestamp'])}, max_ts={int(row['max_timestamp'])}")
else:
    lines.append("Logs: no log entries found in log.csv.")

# Error logs summary
if 'error_count' in globals():
    lines.append(f"Error logs: total rows = {int(error_count)}.")
    if error_count > 0 and 'error_sample' in globals() and not error_sample.empty:
        # show up to 3 sample rows
        sample = error_sample.head(3)
        for i, row in sample.iterrows():
            lines.append(f"  sample {i+1}. ts={int(row['timestamp'])}, cmdb_id={row['cmdb_id']}, message={row['message']}")
    else:
        lines.append("  No sample rows to display (error_logs.csv is empty).")
else:
    lines.append("Error logs: not available.")

# Overall timestamp ranges per file
lines.append("Timestamp ranges (UTC+8):")
for _, r in ranges.iterrows():
    lines.append(f"  {r['file']}: min_ts={r['min_ts']} -> max_ts={r['max_ts']} "
                 f"({r['min_dt_local']} to {r['max_dt_local']})")

summary = "\n".join(lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name  count  min_timestamp  max_timestamp  global_P95_value
0       IG01           JVM-Threads_7778_JVM_ThreadCount_Threads     31     1614857400     1614859200      9.100000e+01
1    Redis01                        OSLinux-CPU_CPU_CPUidleutil     31     1614857400     1614859200      7.382480e+01
2    Mysql02                 OSLinux-OSLinux_ZABBIX_Host_Uptime     31     1614857400     1614859200      6.057396e+06
3    Mysql02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount     31     1614857400     1614859200      4.000000e+00
4    Redis01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT     31     1614857400     1614859200      0.000000e+00
5    Redis01                            OSLinux-CPU_CPU_CPULoad     31     1614857400     1614859200      1.080000e+00
6    Redis01                         OSLinux-CPU_CPU_CPUSysTime     31     1614857400     1614859200      8.712500e-01
7    Redis01                        OSLinux-CPU_CPU_CPUUserTime     31     1614857400     1614859200      2.522555e+01
8    Redis01                             OSLinux-CPU_CPU_CPUWio     31     1614857400     1614859200      9.770000e-02
9    Redis01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...     31     1614857400     1614859200      4.000000e-01
10   Redis01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount     31     1614857400     1614859200      4.000000e+00
11   Redis01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT     31     1614857400     1614859200      0.000000e+00
12   Mysql01                 OSLinux-OSLinux_ZABBIX_Host_Uptime     31     1614857400     1614859200      6.057355e+06
13   Mysql01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...     31     1614857400     1614859200      4.000000e-01
14   Mysql01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount     31     1614857400     1614859200      7.000000e+00
15   Redis01                 OSLinux-OSLinux_ZABBIX_Host_Uptime     31     1614857400     1614859200      6.058517e+06
16   Redis01              redis-Redis_6379_Redis  (aof_enabled)     31     1614857400     1614859200      0.000000e+00
17   Redis01  redis-Redis_6379_Redis  (aof_rewrite_in_progress)     31     1614857400     1614859200      0.000000e+00
18   Redis01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies     31     1614857400     1614859200      1.000000e+00
19  apache01                         OSLinux-CPU_CPU_CPUSysTime     31     1614857400     1614859200      4.486045e-01,    cmdb_id                       trace_name  count  min_timestamp  max_timestamp
0     IG01         trace.self.duration_mean     30     1614857400     1614859140
1     IG01          trace.self.duration_p95     30     1614857400     1614859140
2     IG01             trace.self.row_count     30     1614857400     1614859140
3     IG01  trace.to_Tomcat01.duration_mean     30     1614857400     1614859140
4     IG01   trace.to_Tomcat01.duration_p95     30     1614857400     1614859140
5     IG01      trace.to_Tomcat01.row_count     30     1614857400     1614859140
6     IG01  trace.to_Tomcat02.duration_mean     30     1614857400     1614859140
7     IG01   trace.to_Tomcat02.duration_p95     30     1614857400     1614859140
8     IG01      trace.to_Tomcat02.row_count     30     1614857400     1614859140
9     IG01  trace.to_Tomcat03.duration_mean     30     1614857400     1614859140
10    IG01   trace.to_Tomcat03.duration_p95     30     1614857400     1614859140
11    IG01      trace.to_Tomcat03.row_count     30     1614857400     1614859140
12    IG01  trace.to_Tomcat04.duration_mean     30     1614857400     1614859140
13    IG01   trace.to_Tomcat04.duration_p95     30     1614857400     1614859140
14    IG01      trace.to_Tomcat04.row_count     30     1614857400     1614859140
15    IG02         trace.self.duration_mean     30     1614857400     1614859140
16    IG02          trace.self.duration_p95     30     1614857400     1614859140
17    IG02             trace.self.row_count     30     1614857400     1614859140
18    IG02  trace.to_Tomcat01.duration_mean     30     1614857400     1614859140
19    IG02   trace.to_Tomcat01.duration_p95     30     1614857400     1614859140, Empty DataFrame
Columns: [cmdb_id, log_name, count, min_timestamp, max_timestamp]
Index: [], 0, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [],              file      min_ts      max_ts              min_dt_local              max_dt_local
0      metric.csv  1614857400  1614859200  2021-03-04 19:30:00+0800  2021-03-04 20:00:00+0800
1       trace.csv  1614857400  1614859140  2021-03-04 19:30:00+0800  2021-03-04 19:59:00+0800
2         log.csv        <NA>        <NA>                      <NA>                      <NA>
3  error_logs.csv        <NA>        <NA>                      <NA>                      <NA>)```
```

In [ ]:
# In[2]:

import numpy as np
import pandas as pd
import pytz

# Reuse existing df_metric if present; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv('metric.csv')
    df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce').astype('Int64')

# 1) Compute global P95 and P05 across the full metric series (per cmdb_id, kpi_name)
metric_thresholds = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(global_P95=lambda s: s.dropna().quantile(0.95),
         global_P05=lambda s: s.dropna().quantile(0.05))
    .reset_index()
)

# keep a compact view for display (top 20)
metric_thresholds_head = metric_thresholds.head(20)

# 2) Filter metric.csv to the incident window timestamps between 1614857400 and 1614859200 inclusive
start_ts = 1614857400
end_ts = 1614859200
metric_window = df_metric[(df_metric['timestamp'] >= start_ts) & (df_metric['timestamp'] <= end_ts)].copy()

# 3) Detect anomaly points using global thresholds from metric_thresholds
# Merge thresholds into the windowed data
metric_window_thresh = metric_window.merge(metric_thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# Identify high and low anomaly points
cond_high = metric_window_thresh['value'] >= metric_window_thresh['global_P95']
cond_low = metric_window_thresh['value'] <= metric_window_thresh['global_P05']

high_df = metric_window_thresh[cond_high].copy()
low_df = metric_window_thresh[cond_low].copy()

high_df['anomaly_type'] = 'high'
low_df['anomaly_type'] = 'low'

# Combine anomaly points (a point could appear in both if thresholds equal; we'll keep both records separately)
metric_anomalies = pd.concat([high_df, low_df], ignore_index=True, sort=False)

# Store full anomaly table for later inspection (kernel variable)
# But do not display full table now; provide its shape below.
# 4) Produce compact summary table
if not metric_anomalies.empty:
    agg = (
        metric_anomalies
        .groupby(['cmdb_id', 'kpi_name', 'anomaly_type'], dropna=False)
        .agg(
            anomaly_count=('value', 'count'),
            earliest_anomaly_ts=('timestamp', 'min'),
            latest_anomaly_ts=('timestamp', 'max'),
            max_value=('value', 'max'),
            min_value=('value', 'min'),
            global_P95=('global_P95', 'first'),
            global_P05=('global_P05', 'first'),
        )
        .reset_index()
    )

    # compute severity: high = max_value / global_P95 ; low = global_P05 / min_value
    def compute_severity(row):
        if row['anomaly_type'] == 'high':
            denom = row['global_P95']
            if pd.isna(denom) or denom == 0:
                return np.nan
            return row['max_value'] / denom
        else:
            denom = row['min_value']
            if pd.isna(denom) or denom == 0:
                return np.nan
            return row['global_P05'] / denom

    agg['severity'] = agg.apply(compute_severity, axis=1)

    # Sort by anomaly_count desc, then severity desc
    metric_anomalies_summary = agg.sort_values(['anomaly_count', 'severity'], ascending=[False, False]).reset_index(drop=True)

    # Keep compact top 20 for display
    metric_anomalies_summary_head = metric_anomalies_summary.head(20)

    # total number of distinct component-KPI anomalies (unique cmdb_id,kpi_name pairs with any anomaly)
    distinct_anomaly_components = metric_anomalies[['cmdb_id', 'kpi_name']].drop_duplicates().shape[0]
else:
    # empty results
    metric_anomalies_summary = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','anomaly_type','anomaly_count','earliest_anomaly_ts',
        'latest_anomaly_ts','max_value','min_value','global_P95','global_P05','severity'
    ])
    metric_anomalies_summary_head = metric_anomalies_summary
    distinct_anomaly_components = 0

# Keep outputs compact: show thresholds (head), window shape, anomaly table shape, anomaly summary (head), distinct count
metric_thresholds_head, metric_window.shape, metric_anomalies.shape, metric_anomalies_summary_head, distinct_anomaly_components

```
Out[2]:
```
```python
# Summarize results in plain English using existing kernel variables
summary_lines = []

n_window = int(metric_window.shape[0]) if 'metric_window' in globals() else 0
n_anom = int(metric_anomalies.shape[0]) if 'metric_anomalies' in globals() else 0
distinct = int(distinct_anomaly_components) if 'distinct_anomaly_components' in globals() else 0

summary_lines.append(f"Metric window rows (1614857400–1614859200): {n_window}")
summary_lines.append(f"Anomaly records detected (high+low combined): {n_anom}")
summary_lines.append(f"Distinct component–KPI pairs with any anomaly: {distinct}")
summary_lines.append("Top example anomalies (up to 3):")

if 'metric_anomalies_summary_head' in globals() and not metric_anomalies_summary_head.empty:
    for i, r in metric_anomalies_summary_head.head(3).iterrows():
        summary_lines.append(
            f"  - {r['cmdb_id']} | {r['kpi_name']} | {r['anomaly_type']} | "
            f"count={int(r['anomaly_count'])} | severity={float(r['severity']):.3g}"
        )
else:
    summary_lines.append("  (no anomaly summary rows available)")

summary_lines.append(
    "Observation: many top KPIs show anomaly_count=31 and severity≈1.0, "
    "indicating their values equal the computed P95/P05 (constant series)."
)
summary_lines.append(
    "Actionable note: inspect KPIs with identical P95/P05 and constant values — "
    "these will be flagged as both high and low. Verify whether constant metrics are valid or need different thresholds."
)

summary = "\n".join(summary_lines)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   cmdb_id                                         kpi_name    global_P95    global_P05
0     IG01         JVM-Memory_7778_JVM_Memory_HeapMemoryMax  2.040136e+09  2.040136e+09
1     IG01       JVM-Memory_7778_JVM_Memory_HeapMemoryUsage  4.613570e+01  1.127680e+01
2     IG01        JVM-Memory_7778_JVM_Memory_HeapMemoryUsed  1.007287e+09  2.561107e+08
3     IG01      JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed  8.266114e+08  8.266114e+08
4     IG01        JVM-Operating System_7778_JVM_JVM_CPULoad  2.025000e-01  1.585000e-01
5     IG01                  JVM-Runtime_7778_JVM_JVM_Uptime  1.922364e+06  1.920803e+06
6     IG01         JVM-Threads_7778_JVM_ThreadCount_Threads  9.100000e+01  8.500000e+01
7     IG01                  OSLinux-CPU_CPU-0_SingleCpuUtil  1.147050e+00  7.903000e-01
8     IG01                  OSLinux-CPU_CPU-0_SingleCpuidle  9.925025e+01  9.889451e+01
9     IG01                  OSLinux-CPU_CPU-2_SingleCpuUtil  1.588850e+00  8.652000e-01
10    IG01                  OSLinux-CPU_CPU-2_SingleCpuidle  9.912141e+01  9.827531e+01
11    IG01                       OSLinux-CPU_CPU_CPUCpuUtil  2.584910e+01  2.557890e+01
12    IG01                          OSLinux-CPU_CPU_CPULoad  1.240000e+00  1.000000e+00
13    IG01                       OSLinux-CPU_CPU_CPUSysTime  4.219715e-01  2.681770e-01
14    IG01                      OSLinux-CPU_CPU_CPUUserTime  2.521360e+01  2.512940e+01
15    IG01                           OSLinux-CPU_CPU_CPUWio  2.556685e-01  1.250000e-02
16    IG01                      OSLinux-CPU_CPU_CPUidleutil  7.443090e+01  7.397577e+01
17    IG01    OSLinux-OSLinux_FILESYSTEM_-_FSAvailableSpace  9.954000e+03  9.954000e+03
18    IG01          OSLinux-OSLinux_FILESYSTEM_-_FSCapacity  3.024890e+01  3.024880e+01
19    IG01  OSLinux-OSLinux_FILESYSTEM_-_FSInodeUsedPercent  1.636630e+01  1.636627e+01, (28266, 4), (33123, 7),    cmdb_id                                           kpi_name anomaly_type  anomaly_count  earliest_anomaly_ts  latest_anomaly_ts     max_value     min_value    global_P95    global_P05  severity
0     IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax         high             31           1614857400         1614859200  2.040136e+09  2.040136e+09  2.040136e+09  2.040136e+09       1.0
1     IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax          low             31           1614857400         1614859200  2.040136e+09  2.040136e+09  2.040136e+09  2.040136e+09       1.0
2     IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed         high             31           1614857400         1614859200  8.266114e+08  8.266114e+08  8.266114e+08  8.266114e+08       1.0
3     IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed          low             31           1614857400         1614859200  8.266114e+08  8.266114e+08  8.266114e+08  8.266114e+08       1.0
4     IG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies         high             31           1614857400         1614859200  1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00       1.0
5     IG01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies          low             31           1614857400         1614859200  1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00       1.0
6     IG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...         high             31           1614857400         1614859200  3.000000e-01  3.000000e-01  3.000000e-01  3.000000e-01       1.0
7     IG01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...          low             31           1614857400         1614859200  3.000000e-01  3.000000e-01  3.000000e-01  3.000000e-01       1.0
8     IG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount         high             31           1614857400         1614859200  3.000000e+00  3.000000e+00  3.000000e+00  3.000000e+00       1.0
9     IG01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount          low             31           1614857400         1614859200  3.000000e+00  3.000000e+00  3.000000e+00  3.000000e+00       1.0
10    IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax         high             31           1614857400         1614859200  2.040136e+09  2.040136e+09  2.040136e+09  2.040136e+09       1.0
11    IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax          low             31           1614857400         1614859200  2.040136e+09  2.040136e+09  2.040136e+09  2.040136e+09       1.0
12    IG02        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed         high             31           1614857400         1614859200  8.239351e+08  8.239351e+08  8.239351e+08  8.239351e+08       1.0
13    IG02        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed          low             31           1614857400         1614859200  8.239351e+08  8.239351e+08  8.239351e+08  8.239351e+08       1.0
14    IG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies         high             31           1614857400         1614859200  1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00       1.0
15    IG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies          low             31           1614857400         1614859200  1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00       1.0
16    IG02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount         high             31           1614857400         1614859200  3.000000e+00  3.000000e+00  3.000000e+00  3.000000e+00       1.0
17    IG02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount          low             31           1614857400         1614859200  3.000000e+00  3.000000e+00  3.000000e+00  3.000000e+00       1.0
18    MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax         high             31           1614857400         1614859200  2.040136e+09  2.040136e+09  2.040136e+09  2.040136e+09       1.0
19    MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax          low             31           1614857400         1614859200  2.040136e+09  2.040136e+09  2.040136e+09  2.040136e+09       1.0, 1704)```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing kernel variables: metric_anomalies, metric_thresholds
# Ensure required vars exist
if 'metric_anomalies' not in globals() or metric_anomalies.empty:
    raise ValueError("metric_anomalies is not available or empty in the kernel.")

# 1) Build consecutive anomaly segments per group (gap > 60s -> new segment)
ma = metric_anomalies.copy()
# Ensure timestamp is integer
ma['timestamp'] = pd.to_numeric(ma['timestamp'], errors='coerce').astype('Int64')

# Sort by group and timestamp
ma = ma.sort_values(['cmdb_id', 'kpi_name', 'anomaly_type', 'timestamp']).reset_index(drop=True)

# Compute time diff within each group and assign segment ids
ma['time_diff'] = ma.groupby(['cmdb_id', 'kpi_name', 'anomaly_type'])['timestamp'].diff().fillna(0).astype('Int64')
# new segment when time_diff > 60
ma['new_seg'] = (ma['time_diff'] > 60).astype('int')
ma['seg_idx_within_group'] = ma.groupby(['cmdb_id', 'kpi_name', 'anomaly_type'])['new_seg'].cumsum()

# Create a segment identifier
ma['segment_id'] = (
    ma['cmdb_id'].astype(str) + '||' +
    ma['kpi_name'].astype(str) + '||' +
    ma['anomaly_type'].astype(str) + '||seg:' + ma['seg_idx_within_group'].astype(str)
)

# Aggregate into segments
segments = (
    ma
    .groupby(['cmdb_id', 'kpi_name', 'anomaly_type', 'segment_id'], dropna=False)
    .agg(
        start_ts=('timestamp', 'min'),
        end_ts=('timestamp', 'max'),
        length_minutes=('timestamp', 'count'),  # number of points
        max_value=('value', 'max'),
        min_value=('value', 'min'),
        global_P95=('global_P95', 'first'),
        global_P05=('global_P05', 'first')
    )
    .reset_index()
)

# Compute severity per rules
def compute_sev(row):
    try:
        if row['anomaly_type'] == 'high':
            denom = row['global_P95']
            if pd.isna(denom) or denom == 0:
                return np.nan
            return row['max_value'] / denom
        else:
            denom = row['min_value']
            if pd.isna(denom) or denom == 0:
                return np.nan
            return row['global_P05'] / denom
    except Exception:
        return np.nan

segments['severity'] = segments.apply(compute_sev, axis=1)

# 2) Filter segments: length_minutes >=3 AND severity >=1.5
total_segments_count = int(segments.shape[0])
fault_segments = segments[(segments['length_minutes'] >= 3) & (segments['severity'] >= 1.5)].copy()
filtered_segments_count = int(fault_segments.shape[0])

# 3) Compact summary of filtered segments (top 20) sorted by length_minutes desc then severity desc
fault_segments_summary = (
    fault_segments
    .sort_values(['length_minutes', 'severity'], ascending=[False, False])
    .reset_index(drop=True)
)

fault_segments_summary_head = fault_segments_summary[[
    'cmdb_id','kpi_name','anomaly_type','start_ts','end_ts','length_minutes',
    'max_value','min_value','global_P95','global_P05','severity'
]].head(20)

# 4) Earliest start_ts among filtered segments (if any), and local time conversion (UTC+8)
if filtered_segments_count > 0:
    earliest_filtered_start_ts = int(fault_segments_summary['start_ts'].min())
    tz = pytz.timezone('Asia/Shanghai')
    earliest_filtered_start_local = pd.to_datetime(earliest_filtered_start_ts, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')
else:
    earliest_filtered_start_ts = pd.NA
    earliest_filtered_start_local = pd.NA

# Store full segments table in kernel variable 'segments' and filtered in 'fault_segments'
# Provide compact outputs: head of segments, head of fault_segments_summary, counts, earliest ts
segments_head = segments[[
    'cmdb_id','kpi_name','anomaly_type','start_ts','end_ts','length_minutes','max_value','min_value','global_P95','global_P05','severity'
]].head(20)

fault_segments_summary_head, segments_head, total_segments_count, filtered_segments_count, earliest_filtered_start_ts, earliest_filtered_start_local

```
Out[3]:
```
# Summary in plain English based on the computed kernel variables
summary = (
    f"Segmenting metric anomalies into consecutive segments (gap > 60s) produced {total_segments_count} total segments.\n"
    f"After filtering for segments with length_minutes >= 3 and severity >= 1.5, 0 segments remained (filtered count = {filtered_segments_count}).\n\n"
    "Key observations:\n"
    "- Many segments are single-point (length 1) or full-window length (31 points) but have severity ≈ 1.0 because the metric values equal their global P95/P05.\n"
    "- These constant-series KPIs are flagged as both 'high' and 'low' (P95 == value and P05 == value), producing severity=1.0 and thus were excluded by the severity >= 1.5 filter.\n"
    "- Some segments show mild deviations (severity slightly >1.0) but are short (length 1) and therefore also excluded by the length >= 3 requirement.\n\n"
    "Recommendations:\n"
    "- If you want to surface shorter but meaningful spikes, consider lowering the length threshold (e.g., length >= 2) or reducing the severity cutoff (e.g., >= 1.2).\n"
    "- For metrics that are effectively constant, consider excluding them from threshold-based anomaly detection or use alternative methods (e.g., change detection or absolute delta thresholds).\n"
    "- Inspect the 'segments' and 'metric_thresholds' kernel variables to review examples of segments and the computed global P95/P05 values when refining rules.\n\n"
    "Conclusion: No fault segments met the strict criteria (length >= 3 and severity >= 1.5). Adjust thresholds or review short-duration anomalies for further investigation."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(Empty DataFrame
Columns: [cmdb_id, kpi_name, anomaly_type, start_ts, end_ts, length_minutes, max_value, min_value, global_P95, global_P05, severity]
Index: [],    cmdb_id                                     kpi_name anomaly_type    start_ts      end_ts  length_minutes     max_value     min_value    global_P95    global_P05  severity
0     IG01     JVM-Memory_7778_JVM_Memory_HeapMemoryMax         high  1614857400  1614859200              31  2.040136e+09  2.040136e+09  2.040136e+09  2.040136e+09  1.000000
1     IG01     JVM-Memory_7778_JVM_Memory_HeapMemoryMax          low  1614857400  1614859200              31  2.040136e+09  2.040136e+09  2.040136e+09  2.040136e+09  1.000000
2     IG01   JVM-Memory_7778_JVM_Memory_HeapMemoryUsage         high  1614857640  1614857640               1  4.646540e+01  4.646540e+01  4.613570e+01  1.127680e+01  1.007146
3     IG01   JVM-Memory_7778_JVM_Memory_HeapMemoryUsage         high  1614858480  1614858480               1  4.613570e+01  4.613570e+01  4.613570e+01  1.127680e+01  1.000000
4     IG01   JVM-Memory_7778_JVM_Memory_HeapMemoryUsage          low  1614857760  1614857760               1  1.092770e+01  1.092770e+01  4.613570e+01  1.127680e+01  1.031946
5     IG01   JVM-Memory_7778_JVM_Memory_HeapMemoryUsage          low  1614858600  1614858600               1  1.127680e+01  1.127680e+01  4.613570e+01  1.127680e+01  1.000000
6     IG01    JVM-Memory_7778_JVM_Memory_HeapMemoryUsed         high  1614857640  1614857640               1  1.015131e+09  1.015131e+09  1.007287e+09  2.561107e+08  1.007788
7     IG01    JVM-Memory_7778_JVM_Memory_HeapMemoryUsed         high  1614858480  1614858480               1  1.011375e+09  1.011375e+09  1.007287e+09  2.561107e+08  1.004059
8     IG01    JVM-Memory_7778_JVM_Memory_HeapMemoryUsed          low  1614857700  1614857700               1  2.229402e+08  2.229402e+08  1.007287e+09  2.561107e+08  1.148787
9     IG01    JVM-Memory_7778_JVM_Memory_HeapMemoryUsed          low  1614858540  1614858540               1  2.300614e+08  2.300614e+08  1.007287e+09  2.561107e+08  1.113228
10    IG01  JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed         high  1614857400  1614859200              31  8.266114e+08  8.266114e+08  8.266114e+08  8.266114e+08  1.000000
11    IG01  JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed          low  1614857400  1614859200              31  8.266114e+08  8.266114e+08  8.266114e+08  8.266114e+08  1.000000
12    IG01    JVM-Operating System_7778_JVM_JVM_CPULoad         high  1614858540  1614858540               1  2.025000e-01  2.025000e-01  2.025000e-01  1.585000e-01  1.000000
13    IG01    JVM-Operating System_7778_JVM_JVM_CPULoad         high  1614858840  1614858840               1  2.118000e-01  2.118000e-01  2.025000e-01  1.585000e-01  1.045926
14    IG01    JVM-Operating System_7778_JVM_JVM_CPULoad          low  1614857820  1614857820               1  1.415000e-01  1.415000e-01  2.025000e-01  1.585000e-01  1.120141
15    IG01    JVM-Operating System_7778_JVM_JVM_CPULoad          low  1614859020  1614859020               1  1.585000e-01  1.585000e-01  2.025000e-01  1.585000e-01  1.000000
16    IG01              JVM-Runtime_7778_JVM_JVM_Uptime         high  1614859080  1614859080               1  1.922364e+06  1.922364e+06  1.922364e+06  1.920803e+06  1.000000
17    IG01              JVM-Runtime_7778_JVM_JVM_Uptime         high  1614859200  1614859200               1  1.922483e+06  1.922483e+06  1.922364e+06  1.920803e+06  1.000062
18    IG01              JVM-Runtime_7778_JVM_JVM_Uptime          low  1614857400  1614857400               1  1.920683e+06  1.920683e+06  1.922364e+06  1.920803e+06  1.000062
19    IG01              JVM-Runtime_7778_JVM_JVM_Uptime          low  1614857520  1614857520               1  1.920803e+06  1.920803e+06  1.922364e+06  1.920803e+06  1.000000, 18596, 0, <NA>, <NA>)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing kernel variables: metric_anomalies, metric_thresholds
if 'metric_anomalies' not in globals() or metric_anomalies is None:
    raise ValueError("metric_anomalies is not available in the kernel.")
if 'metric_thresholds' not in globals() or metric_thresholds is None:
    raise ValueError("metric_thresholds is not available in the kernel.")

# Work on a copy
ma = metric_anomalies.copy()

# Ensure timestamp numeric
ma['timestamp'] = pd.to_numeric(ma['timestamp'], errors='coerce').astype('Int64')

# Sort by group and timestamp
ma = ma.sort_values(['cmdb_id', 'kpi_name', 'anomaly_type', 'timestamp']).reset_index(drop=True)

# Compute time diff within each group and mark new segments when gap > 60s
ma['time_diff'] = ma.groupby(['cmdb_id', 'kpi_name', 'anomaly_type'])['timestamp'].diff().fillna(0).astype('Int64')
ma['new_seg'] = (ma['time_diff'] > 60).astype(int)
ma['seg_idx_within_group'] = ma.groupby(['cmdb_id', 'kpi_name', 'anomaly_type'])['new_seg'].cumsum()

# Compose a stable segment identifier
ma['segment_id'] = (
    ma['cmdb_id'].astype(str) + '||' +
    ma['kpi_name'].astype(str) + '||' +
    ma['anomaly_type'].astype(str) + '||seg:' + ma['seg_idx_within_group'].astype(str)
)

# Aggregate into segments
segments = (
    ma
    .groupby(['cmdb_id', 'kpi_name', 'anomaly_type', 'segment_id'], dropna=False)
    .agg(
        start_ts=('timestamp', 'min'),
        end_ts=('timestamp', 'max'),
        length_minutes=('timestamp', 'count'),  # number of points
        max_value=('value', 'max'),
        min_value=('value', 'min'),
        global_P95=('global_P95', 'first'),
        global_P05=('global_P05', 'first')
    )
    .reset_index()
)

# Compute severity: high => max_value / global_P95 ; low => global_P05 / min_value
def compute_severity(row):
    try:
        if row['anomaly_type'] == 'high':
            denom = row['global_P95']
            if pd.isna(denom) or denom == 0:
                return np.nan
            return row['max_value'] / denom
        else:
            denom = row['min_value']
            if pd.isna(denom) or denom == 0:
                return np.nan
            return row['global_P05'] / denom
    except Exception:
        return np.nan

segments['severity'] = segments.apply(compute_severity, axis=1)

# 2) Keep segments with length_minutes >= 2 AND severity >= 1.2
relaxed_fault_segments = segments[(segments['length_minutes'] >= 2) & (segments['severity'] >= 1.2)].copy()

# 3) Compact summary (limit 20 rows) sorted by length_minutes desc then severity desc
relaxed_fault_segments_summary = (
    relaxed_fault_segments
    .sort_values(['length_minutes', 'severity'], ascending=[False, False])
    .reset_index(drop=True)
)

relaxed_fault_segments_summary_head = relaxed_fault_segments_summary[[
    'cmdb_id','kpi_name','anomaly_type','start_ts','end_ts','length_minutes',
    'max_value','min_value','global_P95','global_P05','severity'
]].head(20)

# 4a) total number of relaxed segments found
relaxed_count = int(relaxed_fault_segments.shape[0])

# 4b) per-component count of relaxed segments (top 10 components)
per_component_counts = (
    relaxed_fault_segments
    .groupby('cmdb_id', dropna=False)
    .size()
    .reset_index(name='relaxed_segment_count')
    .sort_values('relaxed_segment_count', ascending=False)
    .head(10)
    .reset_index(drop=True)
)

# Final compact outputs (store full relaxed fault segments in 'relaxed_fault_segments' variable)
relaxed_fault_segments_summary_head, relaxed_count, per_component_counts

```
Out[4]:
```
# Summarize relaxed fault segments using existing kernel variables
lines = []
rc = relaxed_count if 'relaxed_count' in globals() else (relaxed_fault_segments.shape[0] if 'relaxed_fault_segments' in globals() else 0)
lines.append(f"Total relaxed segments found: {rc}")

# Top components by relaxed segment count
if 'per_component_counts' in globals() and not per_component_counts.empty:
    comps = ", ".join(f"{r.cmdb_id}={int(r.relaxed_segment_count)}" for _, r in per_component_counts.iterrows())
    lines.append(f"Top components (by relaxed segments): {comps}")
else:
    lines.append("Top components: (none)")

# Top example segments (up to 5)
lines.append("Example relaxed segments (up to 5):")
if 'relaxed_fault_segments_summary_head' in globals() and not relaxed_fault_segments_summary_head.empty:
    for _, r in relaxed_fault_segments_summary_head.head(5).iterrows():
        lines.append(
            f" - {r.cmdb_id} | {r.kpi_name} | {r.anomaly_type} | "
            f"start={int(r.start_ts)} end={int(r.end_ts)} length={int(r.length_minutes)} severity={float(r.severity):.3f}"
        )
else:
    lines.append(" (no relaxed segments to show)")

# Short interpretation / recommended next steps
lines.append("")
lines.append("Interpretation: Most relaxed segments are short (length=2) spikes with moderate severity (>=1.2).")
lines.append("Recommend reviewing the listed component-KPIs (CPU WIO, CPULoad, app metrics) for transient spikes and "
             "deciding if alerts should include these short-duration anomalies or raise thresholds further.")

summary = "\n".join(lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(          cmdb_id                                           kpi_name anomaly_type    start_ts      end_ts  length_minutes  max_value  min_value  global_P95  global_P05  severity
0         Redis02                             OSLinux-CPU_CPU_CPUWio         high  1614858720  1614858780               2    0.32892   0.095800    0.079100    0.004162  4.158281
1         Redis02                            OSLinux-CPU_CPU_CPULoad         high  1614858180  1614858240               2    0.21000   0.070000    0.070000    0.000000  3.000000
2    ServiceTest7                                            app.cnt          low  1614857640  1614857700               2    8.00000   3.000000   66.500000    8.000000  2.666667
3         Mysql01                             OSLinux-CPU_CPU_CPUWio         high  1614859080  1614859140               2    2.59287   1.092600    0.978500    0.272200  2.649842
4    ServiceTest3                                            app.mrt         high  1614857400  1614857460               2    1.27833   0.794000    0.508370    0.016000  2.514566
5   ServiceTest11                                            app.cnt          low  1614858720  1614858780               2    4.00000   3.000000   72.500000    7.500000  2.500000
6            IG01                             OSLinux-CPU_CPU_CPUWio         high  1614859020  1614859080               2    0.62780   0.370000    0.255668    0.012500  2.455523
7         Redis01                             OSLinux-CPU_CPU_CPUWio         high  1614859020  1614859080               2    0.22860   0.120600    0.097700    0.004200  2.339816
8        Tomcat02  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...         high  1614859020  1614859080               2    2.00000   1.000000    1.000000    0.000000  2.000000
9    ServiceTest8                                            app.cnt          low  1614858960  1614859020               2    8.00000   5.000000   74.500000    9.500000  1.900000
10        Mysql01                         OSLinux-CPU_CPU_CPUCpuUtil         high  1614859140  1614859200               2    3.07490   1.707400    1.707400    0.747800  1.800925
11           IG02                             OSLinux-CPU_CPU_CPUWio         high  1614859020  1614859080               2    0.47800   0.344900    0.290900    0.016600  1.643176
12   ServiceTest5                                            app.mrt         high  1614857400  1614857460               2    0.53183   0.449000    0.335950    0.016555  1.583063
13   ServiceTest9                                            app.cnt          low  1614858060  1614858120               2    9.00000   8.000000   66.500000   12.500000  1.562500
14   ServiceTest9                                            app.mrt         high  1614857400  1614857460               2    0.56822   0.503700    0.373540    0.015700  1.521176
15   ServiceTest4                                            app.mrt         high  1614857400  1614857460               2    1.35900   1.048440    0.899810    0.272985  1.510319
16   ServiceTest6                                            app.mrt         high  1614857400  1614857460               2    0.95525   0.741500    0.643310    0.057235  1.484898
17   ServiceTest8                                            app.mrt         high  1614857400  1614857460               2    1.16364   1.104440    0.794620    0.044250  1.464398
18        Mysql02                             OSLinux-CPU_CPU_CPUWio         high  1614857460  1614857520               2   20.05910  13.993089   13.725749    4.094485  1.461421
19   ServiceTest7                                            app.mrt         high  1614857400  1614857460               2    0.97100   0.765330    0.691355    0.056290  1.404488, 28,         cmdb_id  relaxed_segment_count
0          IG02                      3
1       Redis02                      2
2       Mysql01                      2
3  ServiceTest7                      2
4  ServiceTest8                      2
5  ServiceTest5                      2
6  ServiceTest9                      2
7  ServiceTest3                      2
8          IG01                      1
9          MG02                      1)```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing kernel variables where possible:
# relaxed_fault_segments (full relaxed segments), df_trace or trace.csv
if 'relaxed_fault_segments' not in globals():
    raise ValueError("relaxed_fault_segments not found in kernel. Run prior steps first.")
if 'df_trace' not in globals():
    # try to load trace.csv if not present
    df_trace = pd.read_csv('trace.csv')
    df_trace['timestamp'] = pd.to_numeric(df_trace['timestamp'], errors='coerce').astype('Int64')

# 1) Filter relaxed_fault_segments to allowed root-cause components
allowed_components = [
    'apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
    'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02'
]

filtered_relaxed_candidates = relaxed_fault_segments[relaxed_fault_segments['cmdb_id'].isin(allowed_components)].copy()

# Aggregate per component: relaxed_segment_count, earliest_segment_start_ts, top_kpis (up to 3 examples)
def top_kpis_series(series):
    # take up to 3 distinct kpi names
    uniques = pd.Series(series.unique()).dropna().astype(str)
    return ";".join(uniques.head(3).tolist())

allowed_components_summary = (
    filtered_relaxed_candidates
    .groupby('cmdb_id', dropna=False)
    .agg(
        relaxed_segment_count=('segment_id', 'count'),
        earliest_segment_start_ts=('start_ts', 'min'),
        top_kpis=('kpi_name', lambda s: top_kpis_series(s))
    )
    .reset_index()
    .sort_values(['relaxed_segment_count', 'earliest_segment_start_ts'], ascending=[False, True])
    .reset_index(drop=True)
)

allowed_components_summary_head = allowed_components_summary.head(20)

# 2) Trace anomalies: compute global_P95 across full trace series then detect anomalies in the incident window
# Compute global P95 per (cmdb_id, trace_name) using full df_trace
trace_thresholds = (
    df_trace
    .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index(name='global_P95')
)

# Filter trace.csv to incident window
start_ts = 1614857400
end_ts = 1614859200
trace_window = df_trace[(df_trace['timestamp'] >= start_ts) & (df_trace['timestamp'] <= end_ts)].copy()

# Merge thresholds into windowed trace data
trace_window_thresh = trace_window.merge(trace_thresholds, on=['cmdb_id', 'trace_name'], how='left')

# Detect anomaly points where value >= global_P95 (keep only when global_P95 is not null)
trace_anomalies = trace_window_thresh[
    trace_window_thresh['global_P95'].notna() & (trace_window_thresh['value'] >= trace_window_thresh['global_P95'])
].copy()

# Aggregate anomalies per (cmdb_id, trace_name)
if not trace_anomalies.empty:
    trace_anomaly_summary = (
        trace_anomalies
        .groupby(['cmdb_id', 'trace_name'], dropna=False)
        .agg(
            anomaly_count=('value', 'count'),
            earliest_anomaly_ts=('timestamp', 'min'),
            latest_anomaly_ts=('timestamp', 'max'),
            max_value=('value', 'max'),
            global_P95=('global_P95', 'first')
        )
        .reset_index()
    )
    # compute ratio for sorting (max_value / global_P95), handle zeros/nans
    trace_anomaly_summary['severity_ratio'] = trace_anomaly_summary.apply(
        lambda r: (r['max_value'] / r['global_P95']) if (pd.notna(r['global_P95']) and r['global_P95'] != 0) else np.nan,
        axis=1
    )

    trace_anomaly_summary = trace_anomaly_summary.sort_values(
        ['anomaly_count', 'severity_ratio'], ascending=[False, False]
    ).reset_index(drop=True)
else:
    trace_anomaly_summary = pd.DataFrame(columns=[
        'cmdb_id','trace_name','anomaly_count','earliest_anomaly_ts','latest_anomaly_ts','max_value','global_P95','severity_ratio'
    ])

trace_anomaly_summary_head = trace_anomaly_summary.head(20)

# Store full intermediate tables in kernel variables:
# - filtered_relaxed_candidates (already set)
# - trace_anomalies (full anomaly points)
# Summary tables to display (compact): allowed_components_summary_head, trace_anomaly_summary_head

allowed_components_summary_head, trace_anomaly_summary_head

```
Out[5]:
```
Summary (plain English)

1) Relaxed-segment candidates restricted to allowed root-cause components
- Components with relaxed segments (counts):
  - IG02: 3 segments (earliest start_ts 1614857880) — example KPIs: JVM-Memory_7778_JVM_Memory_HeapMemoryUsage; OSLinux-CPU_CPU-...
  - Redis02: 2 segments (earliest start_ts 1614858180) — KPIs: OSLinux-CPU_CPU_CPULoad; OSLinux-CPU_CPU_CPUWio
  - Mysql01: 2 segments (earliest start_ts 1614859080) — KPIs: OSLinux-CPU_CPU_CPUCpuUtil; OSLinux-CPU_CPU_CPUWio
  - MG02: 1 segment (start_ts 1614857400)
  - Mysql02: 1 segment (start_ts 1614857460)
  - IG01: 1 segment (start_ts 1614859020)
  - Redis01: 1 segment (start_ts 1614859020)
  - Tomcat02: 1 segment (start_ts 1614859020)
- Interpretation: IG02 has the most relaxed segments (3) and should be prioritized, followed by Redis02 and Mysql01. Top KPI examples are included to focus investigation.

2) Trace anomalies (P95-based) within incident window
Top trace anomaly summaries (showing key rows):
- dockerA1 — trace.self.duration_p95: anomaly_count=5, earliest=1614857640, latest=1614858780, max_value=0.0160, global_P95=0.0140, ratio≈1.143
- dockerA2 — trace.self.duration_p95: anomaly_count=5, earliest=1614857580, latest=1614858840, max_value=0.0120, global_P95=0.0120, ratio=1.000
- MG02 — trace.to_dockerA1.row_count: anomaly_count=3, earliest=1614857460, latest=1614858000, max_value=59.0, global_P95=50.0, ratio=1.18
- dockerA1 — trace.from_MG02.row_count: anomaly_count=3, earliest=1614857460, latest=1614858000, max_value=59.0, global_P95=50.0, ratio=1.18
- MG01 — trace.from_dockerA1.row_count: anomaly_count=3, earliest=1614858180, latest=1614858480, max_value=21.0, global_P95=18.0, ratio≈1.167

- Interpretation: Several trace links and self-duration P95 metrics show repeated exceedances (counts 3–5). Notable items to investigate: dockerA1/dockerA2 traces and MG02↔dockerA1 interactions (row_count spikes).

3) Next steps / notes
- Focus first on components with the most relaxed segments (IG02, Redis02, Mysql01) and the trace links with the highest counts/ratios (dockerA1/dockerA2, MG02).
- Stored kernel variables you can inspect further:
  - filtered_relaxed_candidates (relaxed segments filtered to allowed components)
  - relaxed_fault_segments (full relaxed segment table)
  - trace_anomalies (full trace anomaly points) and trace_anomaly_summary (aggregated trace anomaly summary)
- If you want, I can convert the epoch timestamps to readable UTC+8 datetimes, or expand any of the listed segment/trace rows for more detail.

The original code execution output of IPython Kernel is also provided below for reference:

(    cmdb_id  relaxed_segment_count  earliest_segment_start_ts                                           top_kpis
0      IG02                      3                 1614857880  JVM-Memory_7778_JVM_Memory_HeapMemoryUsage;OSL...
1   Redis02                      2                 1614858180     OSLinux-CPU_CPU_CPULoad;OSLinux-CPU_CPU_CPUWio
2   Mysql01                      2                 1614859080  OSLinux-CPU_CPU_CPUCpuUtil;OSLinux-CPU_CPU_CPUWio
3      MG02                      1                 1614857400  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...
4   Mysql02                      1                 1614857460                             OSLinux-CPU_CPU_CPUWio
5      IG01                      1                 1614859020                             OSLinux-CPU_CPU_CPUWio
6   Redis01                      1                 1614859020                             OSLinux-CPU_CPU_CPUWio
7  Tomcat02                      1                 1614859020  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...,      cmdb_id                         trace_name  anomaly_count  earliest_anomaly_ts  latest_anomaly_ts  max_value  global_P95  severity_ratio
0   dockerA1            trace.self.duration_p95              5           1614857640         1614858780     0.0160    0.014000        1.142857
1   dockerA2            trace.self.duration_p95              5           1614857580         1614858840     0.0120    0.012000        1.000000
2       MG02        trace.to_dockerA1.row_count              3           1614857460         1614858000    59.0000   50.000000        1.180000
3   dockerA1          trace.from_MG02.row_count              3           1614857460         1614858000    59.0000   50.000000        1.180000
4       MG01      trace.from_dockerA1.row_count              3           1614858180         1614858480    21.0000   18.000000        1.166667
5   dockerA1            trace.to_MG01.row_count              3           1614858180         1614858480    21.0000   18.000000        1.166667
6       MG02      trace.from_Tomcat02.row_count              3           1614857820         1614859140    70.0000   64.000000        1.093750
7   Tomcat02            trace.to_MG02.row_count              3           1614857820         1614859140    70.0000   64.000000        1.093750
8   dockerB1            trace.self.duration_p95              3           1614857460         1614858120     0.0180    0.017000        1.058824
9       IG01        trace.to_Tomcat04.row_count              3           1614857700         1614859080    65.0000   62.000000        1.048387
10  Tomcat04          trace.from_IG01.row_count              3           1614857700         1614859080    65.0000   62.000000        1.048387
11      MG01        trace.to_dockerA2.row_count              3           1614857460         1614858000    56.0000   54.000000        1.037037
12  dockerA2          trace.from_MG01.row_count              3           1614857460         1614858000    56.0000   54.000000        1.037037
13      IG02               trace.self.row_count              3           1614857400         1614859140   530.0000  520.000000        1.019231
14  dockerB2            trace.self.duration_p95              3           1614857520         1614857760     0.0150    0.015000        1.000000
15      MG01   trace.from_dockerA1.duration_p95              2           1614858000         1614858660     0.5190    0.031920       16.259398
16  dockerA1         trace.to_MG01.duration_p95              2           1614858000         1614858660     0.5190    0.031920       16.259398
17      MG02  trace.from_dockerA2.duration_mean              2           1614857820         1614858900     0.5036    0.050976        9.879198
18  dockerA2        trace.to_MG02.duration_mean              2           1614857820         1614858900     0.5036    0.050976        9.879198
19      MG02   trace.from_dockerA2.duration_p95              2           1614857820         1614858900     1.9838    0.217520        9.120081)```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing kernel variables:
# filtered_relaxed_candidates, trace_anomaly_summary, relaxed_fault_segments
if 'filtered_relaxed_candidates' not in globals():
    raise ValueError("filtered_relaxed_candidates not found in kernel.")
if 'trace_anomaly_summary' not in globals():
    raise ValueError("trace_anomaly_summary not found in kernel.")

# 1) Per-component relaxed segment summary (from filtered_relaxed_candidates)
comp_info = (
    filtered_relaxed_candidates
    .groupby('cmdb_id', dropna=False)
    .agg(
        relaxed_segment_count=('segment_id', 'count'),
        earliest_segment_start_ts=('start_ts', 'min')
    )
    .reset_index()
    .sort_values(['relaxed_segment_count', 'earliest_segment_start_ts'], ascending=[False, True])
    .reset_index(drop=True)
)

# Ensure severity_ratio exists in trace_anomaly_summary
tas = trace_anomaly_summary.copy()
if 'severity_ratio' not in tas.columns:
    tas['severity_ratio'] = tas.apply(
        lambda r: (r['max_value'] / r['global_P95']) if (pd.notna(r.get('global_P95')) and r.get('global_P95') != 0) else np.nan,
        axis=1
    )

# 2) For each component, find up to top 5 matching trace anomalies by severity_ratio desc
rows = []
# iterate components in order from comp_info
for _, comp_row in comp_info.iterrows():
    comp = comp_row['cmdb_id']
    relaxed_count = int(comp_row['relaxed_segment_count'])
    earliest_ts = int(comp_row['earliest_segment_start_ts'])
    # match condition: trace row's cmdb_id equals component OR trace_name contains component string
    mask = (tas['cmdb_id'] == comp) | (tas['trace_name'].astype(str).str.contains(str(comp), na=False))
    matched = tas[mask].copy()
    if matched.empty:
        # still include a row noting no matching traces (as a single row with NaNs for trace fields)
        rows.append({
            'cmdb_id': comp,
            'relaxed_segment_count': relaxed_count,
            'earliest_segment_start_ts': earliest_ts,
            'trace_name': pd.NA,
            'anomaly_count': pd.NA,
            'earliest_anomaly_ts': pd.NA,
            'latest_anomaly_ts': pd.NA,
            'max_value': pd.NA,
            'global_P95': pd.NA,
            'severity_ratio': pd.NA
        })
        continue
    # sort matches by severity_ratio desc, take top 5
    matched_sorted = matched.sort_values('severity_ratio', ascending=False).head(5)
    for _, trow in matched_sorted.iterrows():
        rows.append({
            'cmdb_id': comp,
            'relaxed_segment_count': relaxed_count,
            'earliest_segment_start_ts': earliest_ts,
            'trace_name': trow['trace_name'],
            'anomaly_count': int(trow['anomaly_count']),
            'earliest_anomaly_ts': int(trow['earliest_anomaly_ts']),
            'latest_anomaly_ts': int(trow['latest_anomaly_ts']),
            'max_value': float(trow['max_value']),
            'global_P95': float(trow['global_P95']),
            'severity_ratio': float(trow['severity_ratio'])
        })

# Assemble full correlation table and store in kernel variable
component_trace_correlation = pd.DataFrame(rows)

# 3) Prepare compact display (limit total rows to 20)
component_trace_correlation_display = component_trace_correlation.copy().head(20)

# Also compute total components considered and total matched trace rows
total_components = comp_info.shape[0]
total_matched_rows = component_trace_correlation.shape[0]

# Final compact outputs (stored variables): component_trace_correlation (full), component_trace_correlation_display
component_trace_correlation_display, component_trace_correlation.shape, total_components, total_matched_rows

```
Out[6]:
```
Summary (plain English)

Overview
- We analyzed 8 components from the relaxed metric-fault candidates and matched them to trace anomalies (total matched trace rows = 24). The compact table shows up to 20 rows of those matches.

Key component-level findings
- IG02 (relaxed_segment_count = 3, earliest segment start = 1614857880)
  - Matched high-severity traces (examples, severity_ratio shown):
    - trace.to_Tomcat03.duration_mean — severity ≈ 1.97 (anomaly_count=2, 1614857580–1614857640)
    - trace.from_IG02.duration_mean — severity ≈ 1.97 (same window)
    - trace.to_Tomcat03.duration_p95 — severity ≈ 1.96 (1614857580–1614858660)
    - trace.from_IG02.duration_p95 — severity ≈ 1.96 (1614857580–1614858660)
    - trace.self.duration_mean — severity ≈ 1.53 (1614857400–1614857460)
  - Interpretation: IG02 shows multiple metric segments and several moderate-to-high trace spikes overlapping nearby times — prioritize IG02 investigation.

- MG02 (relaxed_segment_count = 1, earliest = 1614857400)
  - Very high-severity trace matches (examples):
    - trace.from_dockerA2.duration_mean, trace.to_MG02.duration_mean — severity ≈ 9.88 (anomaly_count=2, 1614857820–1614858900)
    - trace.to_MG02.duration_p95, trace.from_dockerA2.duration_p95 — severity ≈ 9.12 and 9.88
    - Additional matched segments with severity ~7.3
  - Interpretation: MG02 trace anomalies have very large severity ratios — strong candidates for root cause due to pronounced trace spikes.

- IG01 (relaxed_segment_count = 1, earliest = 1614859020)
  - Matched traces with moderate severity:
    - trace.to_Tomcat04.duration_mean / from_IG01.duration_mean — severity ≈ 1.456 (1614857460–1614859020)
    - trace.to_Tomcat03/from_IG01.duration_mean — severity ≈ 1.335 (1614857760–1614857820)
    - trace.to_Tomcat04.duration_p95 — severity ≈ 1.331 (1614857460–1614859020)
  - Interpretation: IG01 has metric faults overlapping with moderate trace spikes involving Tomcat destinations.

- Tomcat02 (relaxed_segment_count = 1, earliest = 1614859020)
  - Matched trace: trace.to_MG01.duration_mean — severity ≈ 1.139 (1614857580–1614859140)

- Redis02, Mysql01, Mysql02, Redis01
  - These components had relaxed metric segments (counts shown), but no matching trace anomalies were found in trace_anomaly_summary (no trace rows matched by component name or trace.cmdb_id).

Practical next steps
- Prioritize investigation of MG02 and IG02 first: MG02 has the largest trace severity ratios (>>1) and IG02 has multiple matched traces and several metric segments.
- For each prioritized component, inspect the detailed trace_anomalies and relaxed segment timestamps:
  - MG02: focus on 1614857820–1614858900 windows where duration_p95/mean spikes occurred.
  - IG02: focus on windows around 1614857400–1614858660 and 1614857580–1614857640 where multiple trace metrics spiked.
- Components with NA trace matches (Redis02, Mysql01, Mysql02, Redis01) may still be metric-only issues or require different trace-name matching logic; consider broader trace-name matching or checking other trace metrics.

Stored kernel variables you can inspect for details
- component_trace_correlation (full correlation table)
- filtered_relaxed_candidates (relaxed segments restricted to allowed components)
- trace_anomalies and trace_anomaly_summary (full trace anomaly points and their aggregates)

If you want, I can convert the epoch timestamps above to readable UTC+8 datetimes and/or expand any specific component’s matched trace rows for deeper inspection.

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id  relaxed_segment_count  earliest_segment_start_ts                         trace_name anomaly_count earliest_anomaly_ts latest_anomaly_ts max_value global_P95 severity_ratio
0       IG02                      3                 1614857880    trace.to_Tomcat03.duration_mean             2          1614857580        1614857640  0.400439   0.203048       1.972143
1       IG02                      3                 1614857880      trace.from_IG02.duration_mean             2          1614857580        1614857640  0.400439   0.203048       1.972143
2       IG02                      3                 1614857880     trace.to_Tomcat03.duration_p95             2          1614857580        1614858660     2.266    1.15845       1.956062
3       IG02                      3                 1614857880       trace.from_IG02.duration_p95             2          1614857580        1614858660     2.266    1.15845       1.956062
4       IG02                      3                 1614857880           trace.self.duration_mean             2          1614857400        1614857460  0.840552   0.549635        1.52929
5    Redis02                      2                 1614858180                               <NA>          <NA>                <NA>              <NA>      <NA>       <NA>           <NA>
6    Mysql01                      2                 1614859080                               <NA>          <NA>                <NA>              <NA>      <NA>       <NA>           <NA>
7       MG02                      1                 1614857400  trace.from_dockerA2.duration_mean             2          1614857820        1614858900    0.5036   0.050976       9.879198
8       MG02                      1                 1614857400        trace.to_MG02.duration_mean             2          1614857820        1614858900    0.5036   0.050976       9.879198
9       MG02                      1                 1614857400         trace.to_MG02.duration_p95             2          1614857820        1614858900    1.9838    0.21752       9.120081
10      MG02                      1                 1614857400   trace.from_dockerA2.duration_p95             2          1614857820        1614858900    1.9838    0.21752       9.120081
11      MG02                      1                 1614857400         trace.to_MG02.duration_p95             2          1614857940        1614858000    0.2655    0.03638       7.297966
12   Mysql02                      1                 1614857460                               <NA>          <NA>                <NA>              <NA>      <NA>       <NA>           <NA>
13      IG01                      1                 1614859020    trace.to_Tomcat04.duration_mean             2          1614857460        1614859020  0.286885   0.197058        1.45584
14      IG01                      1                 1614859020      trace.from_IG01.duration_mean             2          1614857460        1614859020  0.286885   0.197058        1.45584
15      IG01                      1                 1614859020    trace.to_Tomcat03.duration_mean             2          1614857760        1614857820  0.259437   0.194283       1.335356
16      IG01                      1                 1614859020      trace.from_IG01.duration_mean             2          1614857760        1614857820  0.259437   0.194283       1.335356
17      IG01                      1                 1614859020     trace.to_Tomcat04.duration_p95             2          1614857460        1614859020     1.947   1.463075       1.330759
18   Redis01                      1                 1614859020                               <NA>          <NA>                <NA>              <NA>      <NA>       <NA>           <NA>
19  Tomcat02                      1                 1614859020        trace.to_MG01.duration_mean             2          1614857580        1614859140  0.158943   0.139491       1.139448, (24, 10), 8, 24)```
```

In [ ]:
# In[7]:

import pandas as pd
import numpy as np
import pytz

# Reuse existing kernel variables: relaxed_fault_segments, trace_anomaly_summary
if 'relaxed_fault_segments' not in globals():
    raise ValueError("relaxed_fault_segments not found in kernel.")
if 'trace_anomaly_summary' not in globals():
    raise ValueError("trace_anomaly_summary not found in kernel.")

# Ensure numeric timestamps
relaxed_fault_segments['start_ts'] = pd.to_numeric(relaxed_fault_segments['start_ts'], errors='coerce').astype('Int64')
relaxed_fault_segments['end_ts'] = pd.to_numeric(relaxed_fault_segments['end_ts'], errors='coerce').astype('Int64')
trace_anomaly_summary['earliest_anomaly_ts'] = pd.to_numeric(trace_anomaly_summary['earliest_anomaly_ts'], errors='coerce').astype('Int64')
trace_anomaly_summary['latest_anomaly_ts'] = pd.to_numeric(trace_anomaly_summary['latest_anomaly_ts'], errors='coerce').astype('Int64')

# Components of interest
components = ['MG02', 'IG02']

# 1) Select segments for MG02 and IG02
segs_sel = relaxed_fault_segments[relaxed_fault_segments['cmdb_id'].isin(components)].copy()

# keep requested columns and sort by start_ts asc, limit top 10 per component
segs_sel = segs_sel[['cmdb_id','kpi_name','anomaly_type','start_ts','end_ts','length_minutes','severity']].sort_values(['cmdb_id','start_ts'], ascending=[True, True])
segs_top_per_comp = segs_sel.groupby('cmdb_id', group_keys=False).head(10).reset_index(drop=True)

# 2) Select trace anomalies matching MG02 or IG02 (by cmdb_id or trace_name contains)
tn = trace_anomaly_summary.copy()
mask = (tn['cmdb_id'].isin(components)) | (tn['trace_name'].astype(str).str.contains('|'.join(components), na=False))
traces_sel = tn[mask].copy()
traces_sel = traces_sel[['cmdb_id','trace_name','anomaly_count','earliest_anomaly_ts','latest_anomaly_ts','max_value','global_P95','severity_ratio']]
traces_sel_sorted = traces_sel.sort_values('severity_ratio', ascending=False).head(20).reset_index(drop=True)

# 3) Find temporal overlaps between segments and traces for each component
matches = []
for comp in components:
    segs_comp = segs_top_per_comp[segs_top_per_comp['cmdb_id'] == comp].copy()
    traces_comp = traces_sel[
        (traces_sel['cmdb_id'] == comp) | (traces_sel['trace_name'].astype(str).str.contains(comp, na=False))
    ].copy()
    if segs_comp.empty or traces_comp.empty:
        continue
    # cross join
    segs_comp['_key'] = 1
    traces_comp['_key'] = 1
    cross = segs_comp.merge(traces_comp, on='_key', suffixes=('_seg','_trace')).drop(columns=['_key'])
    # overlap condition
    cross['overlap'] = (cross['earliest_anomaly_ts'] <= cross['end_ts']) & (cross['latest_anomaly_ts'] >= cross['start_ts'])
    cross = cross[cross['overlap']].copy()
    if cross.empty:
        continue
    # compute overlap_timestamp
    cross['overlap_timestamp'] = cross[['start_ts','earliest_anomaly_ts']].max(axis=1)
    # keep relevant columns and store
    for _, r in cross.iterrows():
        matches.append({
            'cmdb_id': comp,
            'kpi_name': r['kpi_name'],
            'segment_start_ts': int(r['start_ts']),
            'segment_end_ts': int(r['end_ts']),
            'segment_severity': float(r['severity']),
            'matched_trace_name': r['trace_name'],
            'trace_earliest_anomaly_ts': int(r['earliest_anomaly_ts']),
            'trace_latest_anomaly_ts': int(r['latest_anomaly_ts']),
            'trace_severity_ratio': float(r['severity_ratio']),
            'overlap_timestamp': int(r['overlap_timestamp'])
        })

# Full matching pairs table stored in kernel var
component_overlap_matches = pd.DataFrame(matches)

# For each component, choose the earliest overlapping pair (min overlap_timestamp)
if not component_overlap_matches.empty:
    chosen = component_overlap_matches.sort_values(['cmdb_id','overlap_timestamp'], ascending=[True, True]).groupby('cmdb_id', as_index=False).first()
else:
    # empty structure with desired columns
    chosen = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','segment_start_ts','segment_end_ts','segment_severity',
        'matched_trace_name','trace_earliest_anomaly_ts','trace_latest_anomaly_ts',
        'trace_severity_ratio','overlap_timestamp'
    ])

# Prepare compact displays
# step1 display: segs_top_per_comp (selected columns) - top 10 per comp
segs_display = segs_top_per_comp[['cmdb_id','kpi_name','anomaly_type','start_ts','end_ts','length_minutes','severity']].reset_index(drop=True)

# step2 display: traces_sel_sorted (already limited to top 20 by severity_ratio desc)
traces_display = traces_sel_sorted

# step3 compact output: chosen earliest overlapping pair per component (one row per component)
overlap_display = chosen[[
    'cmdb_id','kpi_name','segment_start_ts','segment_end_ts','segment_severity',
    'matched_trace_name','trace_earliest_anomaly_ts','trace_latest_anomaly_ts',
    'trace_severity_ratio','overlap_timestamp'
]].reset_index(drop=True)

# Return compact displays (variables) for the Controller to read
segs_display, traces_display, overlap_display, component_overlap_matches.shape

```
Out[7]:
```
Summary — earliest metric–trace overlaps for MG02 and IG02

1) Metric segments (top per component)
- IG02 segments (sorted by start_ts):
  - JVM-Memory_...HeapMemoryUsage (low) | 1614857880–1614857940 | length=2 | severity=1.254
  - OSLinux-CPU_CPU_CPULoad (high)     | 1614858900–1614858960 | length=2 | severity=1.273
  - OSLinux-CPU_CPU_CPUWio (high)      | 1614859020–1614859080 | length=2 | severity=1.643

- MG02 segments (top entries):
  - OSLinux-OSLinux_PROCESS_zabbix-... (high) | 1614857400–1614857460 | length=2 | severity=1.333

2) Matching trace anomalies (top matches by severity)
- MG02-related traces (very high severity ratios):
  - trace.to_MG02.duration_mean / trace.from_dockerA2.duration_mean | 1614857820–1614858900 | max=0.5036 | global_P95=0.050976 | ratio≈9.88
  - trace.to_MG02.duration_p95 / trace.from_dockerA2.duration_p95 | 1614857820–1614858900 | max=1.9838 | global_P95=0.21752  | ratio≈9.12
  - Additional MG02 matches: interactions with dockerA1, dockerB2 and Tomcat03 with ratios in ~2–7 range.

- IG02-related traces (moderate severity):
  - trace.to_Tomcat03.duration_mean / from_IG02.duration_mean | 1614857580–1614857640 | max≈0.4004 | global_P95≈0.2030 | ratio≈1.97
  - trace.to_Tomcat03.duration_p95 / from_IG02.duration_p95 | 1614857580–1614858660 | max=2.2660 | global_P95≈1.15845 | ratio≈1.96
  - trace.self.duration_mean (IG02) | 1614857400–1614857460 | ratio≈1.53

3) Earliest overlapping metric–trace pair per component (computed)
- MG02 (earliest overlap):
  - Metric segment: OSLinux_...zabbix_agentd (high) | segment 1614857400–1614857460 | severity=1.333
  - Matched trace: trace.to_dockerA2.duration_p95 | trace window 1614857400–1614857520 | trace severity_ratio ≈ 1.387
  - Earliest overlap timestamp: 1614857400
  - Interpretation: MG02’s earliest metric fault coincides with a trace spike starting at the same second (strong temporal correlation); note there are later MG02 traces with much larger ratios (~9x) in 1614857820–1614858900 window.

- IG02 (earliest overlap):
  - Metric segment: HeapMemoryUsage (low) | segment 1614857880–1614857940 | severity=1.254
  - Matched trace: trace.self.row_count (IG02) | trace window 1614857400–1614859140 | trace severity_ratio ≈ 1.019
  - Earliest overlap timestamp: 1614857880
  - Interpretation: IG02’s earliest overlap is with a long trace window (trace.self.row_count) but the trace severity is low (~1.02). However, IG02 also has higher-severity trace spikes (to_Tomcat03) earlier/later (around 1614857580–1614858660) that may overlap other metric segments—these should be checked as well.

Overall conclusion and recommended next steps
- MG02 shows the clearest early correlation: a metric segment at 1614857400 overlaps a trace spike beginning at the same time, and many MG02 traces have very high severity ratios later (strong candidate root cause).
- IG02 has multiple metric segments and moderate trace spikes; earliest overlap trace severity is weak, but higher-severity IG02→Tomcat03 traces exist in the window and warrant attention.
- Next steps: inspect detailed trace spans and metric samples around:
  - MG02: 1614857400–1614858900 (focus on dockerA2, dockerA1, dockerB2 interactions).
  - IG02: 1614857580–1614858660 and around 1614857880 (Tomcat03 interactions and HeapMemoryUsage).
- Full matching pairs are stored in kernel variable component_overlap_matches for further drill-down.

The original code execution output of IPython Kernel is also provided below for reference:

(  cmdb_id                                           kpi_name anomaly_type    start_ts      end_ts  length_minutes  severity
0    IG02         JVM-Memory_7778_JVM_Memory_HeapMemoryUsage          low  1614857880  1614857940               2  1.254063
1    IG02                            OSLinux-CPU_CPU_CPULoad         high  1614858900  1614858960               2  1.272727
2    IG02                             OSLinux-CPU_CPU_CPUWio         high  1614859020  1614859080               2  1.643176
3    MG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...         high  1614857400  1614857460               2  1.333333,      cmdb_id                         trace_name  anomaly_count  earliest_anomaly_ts  latest_anomaly_ts  max_value  global_P95  severity_ratio
0   dockerA2        trace.to_MG02.duration_mean              2           1614857820         1614858900   0.503600    0.050976        9.879198
1       MG02  trace.from_dockerA2.duration_mean              2           1614857820         1614858900   0.503600    0.050976        9.879198
2   dockerA2         trace.to_MG02.duration_p95              2           1614857820         1614858900   1.983800    0.217520        9.120081
3       MG02   trace.from_dockerA2.duration_p95              2           1614857820         1614858900   1.983800    0.217520        9.120081
4       MG02   trace.from_dockerA1.duration_p95              2           1614857940         1614858000   0.265500    0.036380        7.297966
5   dockerA1         trace.to_MG02.duration_p95              2           1614857940         1614858000   0.265500    0.036380        7.297966
6       MG02   trace.from_dockerB2.duration_p95              2           1614858660         1614858900   1.171350    0.174060        6.729576
7   dockerB2         trace.to_MG02.duration_p95              2           1614858660         1614858900   1.171350    0.174060        6.729576
8       MG02  trace.from_dockerA1.duration_mean              2           1614857940         1614858000   0.057455    0.017000        3.379706
9   dockerA1        trace.to_MG02.duration_mean              2           1614857940         1614858000   0.057455    0.017000        3.379706
10      MG02  trace.from_dockerB2.duration_mean              2           1614858660         1614858900   0.144563    0.061442        2.352852
11  dockerB2        trace.to_MG02.duration_mean              2           1614858660         1614858900   0.144563    0.061442        2.352852
12      IG02    trace.to_Tomcat03.duration_mean              2           1614857580         1614857640   0.400439    0.203048        1.972143
13  Tomcat03      trace.from_IG02.duration_mean              2           1614857580         1614857640   0.400439    0.203048        1.972143
14      IG02     trace.to_Tomcat03.duration_p95              2           1614857580         1614858660   2.266000    1.158450        1.956062
15  Tomcat03       trace.from_IG02.duration_p95              2           1614857580         1614858660   2.266000    1.158450        1.956062
16      MG02  trace.from_Tomcat03.duration_mean              2           1614857580         1614857820   0.330957    0.182754        1.810941
17  Tomcat03        trace.to_MG02.duration_mean              2           1614857580         1614857820   0.330957    0.182754        1.810941
18      MG02   trace.from_Tomcat03.duration_p95              2           1614857580         1614857820   1.958500    1.091505        1.794312
19  Tomcat03         trace.to_MG02.duration_p95              2           1614857580         1614857820   1.958500    1.091505        1.794312,   cmdb_id                                           kpi_name  segment_start_ts  segment_end_ts  segment_severity              matched_trace_name  trace_earliest_anomaly_ts  trace_latest_anomaly_ts  trace_severity_ratio  overlap_timestamp
0    IG02         JVM-Memory_7778_JVM_Memory_HeapMemoryUsage        1614857880      1614857940          1.254063            trace.self.row_count                 1614857400               1614859140              1.019231         1614857880
1    MG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...        1614857400      1614857460          1.333333  trace.to_dockerA2.duration_p95                 1614857400               1614857520              1.387169         1614857400, (70, 10))```
```

In [ ]:
# In[8]:

import pandas as pd
import pytz
import numpy as np

# Reuse existing kernel variables where possible
# relaxed_fault_segments, metric_thresholds, metric_window or df_metric, component_overlap_matches, trace_anomaly_summary

tz = pytz.timezone('Asia/Shanghai')

# Ensure metric data available
try:
    metric_window
except NameError:
    try:
        df_metric = pd.read_csv('metric.csv')
        df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce').astype('Int64')
        metric_window = df_metric  # fall back to full metric if window not present
    except Exception as e:
        raise RuntimeError("metric data not available") from e

# Ensure metric_thresholds exists
if 'metric_thresholds' not in globals():
    raise RuntimeError("metric_thresholds not found in kernel.")

# Ensure relaxed_fault_segments exists
if 'relaxed_fault_segments' not in globals():
    raise RuntimeError("relaxed_fault_segments not found in kernel.")

# Ensure trace_anomaly_summary exists
if 'trace_anomaly_summary' not in globals():
    raise RuntimeError("trace_anomaly_summary not found in kernel.")

# Ensure component_overlap_matches exists (may be empty)
if 'component_overlap_matches' not in globals():
    component_overlap_matches = pd.DataFrame()

components = ['MG02', 'IG02']

metric_rows = []
trace_rows = []

for comp in components:
    # 1) earliest relaxed segment start_ts for component
    comp_segs = relaxed_fault_segments[relaxed_fault_segments['cmdb_id'] == comp].copy()
    if comp_segs.empty:
        continue
    earliest_start = int(comp_segs['start_ts'].min())
    # find segments at that start_ts (could be multiple KPIs)
    segs_at_start = comp_segs[comp_segs['start_ts'] == earliest_start].copy()
    # 2) For that start_ts, extract metric rows from metric_window (or df_metric) matching comp and timestamp == start_ts
    for _, seg in segs_at_start.iterrows():
        kpi = seg['kpi_name']
        seg_sev = float(seg['severity'])
        # search in metric_window first
        mrows = metric_window[
            (metric_window['cmdb_id'] == comp) &
            (metric_window['kpi_name'] == kpi) &
            (metric_window['timestamp'] == earliest_start)
        ].copy()
        # fallback to full df_metric if empty (metric_window may already be full)
        if mrows.empty and 'df_metric' in globals():
            mrows = df_metric[
                (df_metric['cmdb_id'] == comp) &
                (df_metric['kpi_name'] == kpi) &
                (df_metric['timestamp'] == earliest_start)
            ].copy()
        # if still empty, create a placeholder row noting missing metric sample
        if mrows.empty:
            metric_rows.append({
                'cmdb_id': comp,
                'kpi_name': kpi,
                'timestamp': earliest_start,
                'value': pd.NA,
                'global_P95': pd.NA,
                'global_P05': pd.NA,
                'segment_severity': seg_sev
            })
        else:
            # merge thresholds
            thr = metric_thresholds[
                (metric_thresholds['cmdb_id'] == comp) &
                (metric_thresholds['kpi_name'] == kpi)
            ]
            gp95 = thr['global_P95'].iloc[0] if not thr.empty else pd.NA
            gp05 = thr['global_P05'].iloc[0] if not thr.empty else pd.NA
            for _, mr in mrows.iterrows():
                metric_rows.append({
                    'cmdb_id': comp,
                    'kpi_name': kpi,
                    'timestamp': int(mr['timestamp']),
                    'value': float(mr['value']) if pd.notna(mr['value']) else pd.NA,
                    'global_P95': gp95,
                    'global_P05': gp05,
                    'segment_severity': seg_sev
                })

    # 3) For same component, extract trace_anomaly_summary rows that were found to overlap that segment
    # First try component_overlap_matches entries matching component and segment_start_ts == earliest_start
    matches = pd.DataFrame()
    if not component_overlap_matches.empty:
        matches = component_overlap_matches[
            (component_overlap_matches['cmdb_id'] == comp) &
            (component_overlap_matches['segment_start_ts'] == earliest_start)
        ].copy()
    # If no matches found, fallback to any trace_anomaly_summary rows whose time window includes the segment start_ts
    if matches.empty:
        # find traces where (trace.cmdb_id == comp OR trace_name contains comp) AND earliest_anomaly_ts <= earliest_start <= latest_anomaly_ts
        mask = (
            ((trace_anomaly_summary['cmdb_id'] == comp) | (trace_anomaly_summary['trace_name'].astype(str).str.contains(comp, na=False))) &
            (trace_anomaly_summary['earliest_anomaly_ts'] <= earliest_start) &
            (trace_anomaly_summary['latest_anomaly_ts'] >= earliest_start)
        )
        matches = trace_anomaly_summary[mask].copy()
    # collect up to 6 trace rows total across components (we'll limit when displaying)
    for _, tr in matches.iterrows():
        trace_rows.append({
            'component_of_interest': comp,
            'trace_cmdb_id': tr.get('cmdb_id', pd.NA),
            'trace_name': tr.get('trace_name', pd.NA),
            'anomaly_count': int(tr.get('anomaly_count')) if pd.notna(tr.get('anomaly_count')) else pd.NA,
            'earliest_anomaly_ts': int(tr.get('earliest_anomaly_ts')) if pd.notna(tr.get('earliest_anomaly_ts')) else pd.NA,
            'latest_anomaly_ts': int(tr.get('latest_anomaly_ts')) if pd.notna(tr.get('latest_anomaly_ts')) else pd.NA,
            'max_value': float(tr.get('max_value')) if pd.notna(tr.get('max_value')) else pd.NA,
            'global_P95': float(tr.get('global_P95')) if pd.notna(tr.get('global_P95')) else pd.NA,
            'severity_ratio': float(tr.get('severity_ratio')) if pd.notna(tr.get('severity_ratio')) else pd.NA
        })

# Assemble DataFrames
metric_df = pd.DataFrame(metric_rows)
trace_df = pd.DataFrame(trace_rows)

# Store full intermediate matches in kernel variable
final_verification_rows = {'metrics': metric_df.copy(), 'traces': trace_df.copy()}

# 4) Convert timestamps to readable UTC+8 datetimes for display
def to_local_str(ts):
    if pd.isna(ts):
        return pd.NA
    return pd.to_datetime(int(ts), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')

if not metric_df.empty:
    metric_df['timestamp_local'] = metric_df['timestamp'].apply(lambda x: to_local_str(x))
if not trace_df.empty:
    trace_df['earliest_anomaly_local'] = trace_df['earliest_anomaly_ts'].apply(lambda x: to_local_str(x))
    trace_df['latest_anomaly_local'] = trace_df['latest_anomaly_ts'].apply(lambda x: to_local_str(x))

# 5) Prepare compact displays with limits: at most 3 metric rows and 6 trace rows
metrics_display = metric_df[[
    'cmdb_id','kpi_name','timestamp','timestamp_local','value','global_P95','global_P05','segment_severity'
]].head(3).reset_index(drop=True)

traces_display = trace_df[[
    'component_of_interest','trace_cmdb_id','trace_name','anomaly_count',
    'earliest_anomaly_ts','earliest_anomaly_local','latest_anomaly_ts','latest_anomaly_local',
    'max_value','global_P95','severity_ratio'
]].sort_values('severity_ratio', ascending=False).head(6).reset_index(drop=True)

# Return compact displays and keep full intermediate in final_verification_rows
metrics_display, traces_display, final_verification_rows

```
Out[8]:
```
Summary — verification for MG02 and IG02 (compact)

Metrics at earliest relaxed-segment start:

- MG02
  - Earliest relaxed segment start_ts: 1614857400 → 2021-03-04 19:30:00+0800
  - Metric row(s):
    - kpi_name: OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_... 
    - timestamp: 1614857400 (2021-03-04 19:30:00+0800)
    - value: 0.300
    - global_P95: 0.3000
    - global_P05: 0.3000
    - segment_severity: 1.333333

- IG02
  - Earliest relaxed segment start_ts: 1614857880 → 2021-03-04 19:38:00+0800
  - Metric row(s):
    - kpi_name: JVM-Memory_7778_JVM_Memory_HeapMemoryUsage
    - timestamp: 1614857880 (2021-03-04 19:38:00+0800)
    - value: 12.502
    - global_P95: 48.8105
    - global_P05: 15.6783
    - segment_severity: 1.254063

Trace evidence for those exact segment start timestamps:

- The trace lookup for traces overlapping the exact segment start_ts returned no concrete trace rows in the compact extracted table (trace results are NA for the direct exact-start matches shown). 
- Note: earlier analysis did identify MG02 and IG02 trace anomalies in nearby windows (e.g., MG02 trace spikes around 1614857820–1614858900 and IG02→Tomcat03 spikes around 1614857580–1614858660). Those broader trace anomalies are stored in kernel variables (component_overlap_matches and trace_anomaly_summary) for review.

Interpretation and recommendation:

- MG02: metric sample at 19:30 shows a value equal to its computed thresholds (P95/P05) and a segment severity reported 1.33. Although the immediate trace extraction at that exact second returned no trace row here, previous trace analysis shows very large MG02-related trace spikes shortly afterward (e.g., starting ~1614857820). Suggest inspecting component_overlap_matches and trace_anomalies (especially MG02 trace windows 1614857820–1614858900) to confirm temporal correlation and precise causality.
- IG02: metric sample at 19:38 shows a low anomaly (value below global_P05) with severity ≈1.25. The compact trace lookup at that exact timestamp did not return a matching trace row, but moderate-to-strong IG02 trace spikes (to/from Tomcat03) were seen in nearby windows. Inspect those trace windows (around 1614857580–1614858660) for overlap with other IG02 metric segments.

Where to get full details (already stored in kernel):
- final_verification_rows (contains full metric and trace rows collected for these checks)
- component_overlap_matches (full matching pairs)
- trace_anomaly_summary and trace_anomalies (full trace aggregates and points)

If you want, I can:
- show the component_overlap_matches rows for MG02 and IG02, or
- expand final_verification_rows['traces'] / final_verification_rows['metrics'] (head) to inspect the raw matching evidence with exact trace windows and times.

The original code execution output of IPython Kernel is also provided below for reference:

(  cmdb_id                                           kpi_name   timestamp           timestamp_local   value  global_P95  global_P05  segment_severity
0    MG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  1614857400  2021-03-04 19:30:00+0800   0.300      0.3000      0.3000          1.333333
1    IG02         JVM-Memory_7778_JVM_Memory_HeapMemoryUsage  1614857880  2021-03-04 19:38:00+0800  12.502     48.8105     15.6783          1.254063,   component_of_interest trace_cmdb_id trace_name anomaly_count earliest_anomaly_ts earliest_anomaly_local latest_anomaly_ts latest_anomaly_local max_value global_P95 severity_ratio
0                  MG02          MG02       <NA>          <NA>                <NA>                   <NA>              <NA>                 <NA>      <NA>       <NA>           <NA>
1                  MG02          MG02       <NA>          <NA>                <NA>                   <NA>              <NA>                 <NA>      <NA>       <NA>           <NA>
2                  MG02          MG02       <NA>          <NA>                <NA>                   <NA>              <NA>                 <NA>      <NA>       <NA>           <NA>
3                  MG02          MG02       <NA>          <NA>                <NA>                   <NA>              <NA>                 <NA>      <NA>       <NA>           <NA>
4                  MG02          MG02       <NA>          <NA>                <NA>                   <NA>              <NA>                 <NA>      <NA>       <NA>           <NA>
5                  MG02          MG02       <NA>          <NA>                <NA>                   <NA>              <NA>                 <NA>      <NA>       <NA>           <NA>, {'metrics':   cmdb_id                                           kpi_name   timestamp   value  global_P95  global_P05  segment_severity
0    MG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  1614857400   0.300      0.3000      0.3000          1.333333
1    IG02         JVM-Memory_7778_JVM_Memory_HeapMemoryUsage  1614857880  12.502     48.8105     15.6783          1.254063, 'traces':    component_of_interest trace_cmdb_id trace_name anomaly_count earliest_anomaly_ts latest_anomaly_ts max_value global_P95 severity_ratio
0                   MG02          MG02       <NA>          <NA>                <NA>              <NA>      <NA>       <NA>           <NA>
1                   MG02          MG02       <NA>          <NA>                <NA>              <NA>      <NA>       <NA>           <NA>
2                   MG02          MG02       <NA>          <NA>                <NA>              <NA>      <NA>       <NA>           <NA>
3                   MG02          MG02       <NA>          <NA>                <NA>              <NA>      <NA>       <NA>           <NA>
4                   MG02          MG02       <NA>          <NA>                <NA>              <NA>      <NA>       <NA>           <NA>
..                   ...           ...        ...           ...                 ...               ...       ...        ...            ...
43                  IG02          IG02       <NA>          <NA>                <NA>              <NA>      <NA>       <NA>           <NA>
44                  IG02          IG02       <NA>          <NA>                <NA>              <NA>      <NA>       <NA>           <NA>
45                  IG02          IG02       <NA>          <NA>                <NA>              <NA>      <NA>       <NA>           <NA>
46                  IG02          IG02       <NA>          <NA>                <NA>              <NA>      <NA>       <NA>           <NA>
47                  IG02          IG02       <NA>          <NA>                <NA>              <NA>      <NA>       <NA>           <NA>

[48 rows x 9 columns]})

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[9]:

import pandas as pd
import pytz
import numpy as np

# Reuse kernel variables
# relaxed_fault_segments, metric_thresholds, metric_window (or df_metric), trace_anomaly_summary
tz = pytz.timezone('Asia/Shanghai')

# Ensure required variables exist
for v in ['relaxed_fault_segments', 'metric_thresholds', 'trace_anomaly_summary']:
    if v not in globals():
        raise RuntimeError(f"{v} not found in kernel.")

# Ensure metric_window or df_metric available
if 'metric_window' in globals():
    metrics_source = metric_window
elif 'df_metric' in globals():
    metrics_source = df_metric
else:
    # try to load metric.csv as fallback
    metrics_source = pd.read_csv('metric.csv')
    metrics_source['timestamp'] = pd.to_numeric(metrics_source['timestamp'], errors='coerce').astype('Int64')

components = ['MG02', 'IG02']

# 1) For each component get earliest relaxed segment start_ts
earliest_start = (
    relaxed_fault_segments[relaxed_fault_segments['cmdb_id'].isin(components)]
    .groupby('cmdb_id', dropna=False)['start_ts']
    .min()
    .reset_index()
    .rename(columns={'start_ts': 'earliest_start_ts'})
)

# helper to convert ts -> local string
def ts_local_str(x):
    if pd.isna(x):
        return pd.NA
    return pd.to_datetime(int(x), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S%z')

metric_rows = []
trace_rows = []

# Pre-index thresholds for fast lookup
thr = metric_thresholds.copy()

for _, r in earliest_start.iterrows():
    comp = r['cmdb_id']
    start_ts = int(r['earliest_start_ts'])
    # 2) extract metric row(s) matching component and timestamp == start_ts
    segs = relaxed_fault_segments[
        (relaxed_fault_segments['cmdb_id'] == comp) &
        (relaxed_fault_segments['start_ts'] == start_ts)
    ].copy()
    # For each segment (kpi) at that start, find metric row in metrics_source
    for _, s in segs.iterrows():
        kpi = s['kpi_name']
        seg_sev = float(s['severity'])
        m_rows = metrics_source[
            (metrics_source['cmdb_id'] == comp) &
            (metrics_source['kpi_name'] == kpi) &
            (metrics_source['timestamp'] == start_ts)
        ].copy()
        if m_rows.empty:
            # placeholder if missing
            metric_rows.append({
                'cmdb_id': comp,
                'kpi_name': kpi,
                'timestamp': start_ts,
                'timestamp_local': ts_local_str(start_ts),
                'value': pd.NA,
                'global_P95': pd.NA,
                'global_P05': pd.NA,
                'segment_severity': seg_sev
            })
        else:
            # merge thresholds for this kpi
            trow = thr[(thr['cmdb_id'] == comp) & (thr['kpi_name'] == kpi)]
            gp95 = trow['global_P95'].iloc[0] if not trow.empty else pd.NA
            gp05 = trow['global_P05'].iloc[0] if not trow.empty else pd.NA
            for _, mr in m_rows.iterrows():
                metric_rows.append({
                    'cmdb_id': comp,
                    'kpi_name': kpi,
                    'timestamp': int(mr['timestamp']),
                    'timestamp_local': ts_local_str(mr['timestamp']),
                    'value': float(mr['value']) if pd.notna(mr['value']) else pd.NA,
                    'global_P95': gp95,
                    'global_P05': gp05,
                    'segment_severity': seg_sev
                })

    # 3) Extract trace rows overlapping either the exact segment time or broader windows:
    if comp == 'MG02':
        win_start, win_end = 1614857400, 1614858900
    else:  # IG02
        win_start, win_end = 1614857400, 1614859140

    ta = trace_anomaly_summary.copy()
    # overlap condition with broader window
    mask_win = (ta['earliest_anomaly_ts'] <= win_end) & (ta['latest_anomaly_ts'] >= win_start)
    # restrict to traces owned by comp or containing comp in trace_name
    mask_comp = (ta['cmdb_id'] == comp) | (ta['trace_name'].astype(str).str.contains(comp, na=False))
    matched = ta[mask_win & mask_comp].copy()

    # convert timestamps for display later
    if not matched.empty:
        matched = matched.assign(
            earliest_local = matched['earliest_anomaly_ts'].apply(lambda x: ts_local_str(x)),
            latest_local = matched['latest_anomaly_ts'].apply(lambda x: ts_local_str(x))
        )
        # collect trace info rows; will sort and limit later
        for _, tr in matched.iterrows():
            trace_rows.append({
                'component': comp,
                'trace_cmdb_id': tr['cmdb_id'],
                'trace_name': tr['trace_name'],
                'anomaly_count': int(tr['anomaly_count']),
                'earliest_anomaly_ts': int(tr['earliest_anomaly_ts']),
                'earliest_anomaly_local': ts_local_str(tr['earliest_anomaly_ts']),
                'latest_anomaly_ts': int(tr['latest_anomaly_ts']),
                'latest_anomaly_local': ts_local_str(tr['latest_anomaly_ts']),
                'max_value': float(tr['max_value']),
                'global_P95': float(tr['global_P95']) if pd.notna(tr['global_P95']) else pd.NA,
                'severity_ratio': float(tr['severity_ratio']) if pd.notna(tr['severity_ratio']) else pd.NA
            })

# Assemble DataFrames
metrics_df = pd.DataFrame(metric_rows)
traces_df = pd.DataFrame(trace_rows)

# 4) Limit rows to respect overall compact cap (max 10 rows total).
# We will include all metric rows (likely small) and allocate remaining slots to traces,
# preferring higher severity_ratio per component.
max_total_rows = 10
metrics_count = len(metrics_df)
remaining = max(0, max_total_rows - metrics_count)

# Partition traces by component and pick top by severity_ratio
final_trace_rows = []
if not traces_df.empty and remaining > 0:
    comps_in_traces = traces_df['component'].unique().tolist()
    # allocate per-component quota fairly
    per_comp_quota = remaining // max(1, len(comps_in_traces))
    extra = remaining % max(1, len(comps_in_traces))
    for comp in comps_in_traces:
        comp_rows = traces_df[traces_df['component'] == comp].sort_values('severity_ratio', ascending=False)
        quota = per_comp_quota + (1 if extra > 0 else 0)
        if extra > 0:
            extra -= 1
        selected = comp_rows.head(quota)
        final_trace_rows.append(selected)
    if final_trace_rows:
        final_traces_df = pd.concat(final_trace_rows, ignore_index=True)
    else:
        final_traces_df = pd.DataFrame(columns=traces_df.columns)
else:
    final_traces_df = pd.DataFrame(columns=traces_df.columns)

# Prepare final_evidence kernel variable (full collected sets, but compact display below)
final_evidence = {
    'metrics': metrics_df.copy(),       # full metric rows collected
    'traces': traces_df.copy()          # full trace matches collected
}

# Prepare compact display tables
metrics_display = metrics_df[[
    'cmdb_id','kpi_name','timestamp','timestamp_local','value','global_P95','global_P05','segment_severity'
]].reset_index(drop=True)

traces_display = final_traces_df[[
    'component','trace_cmdb_id','trace_name','anomaly_count',
    'earliest_anomaly_ts','earliest_anomaly_local','latest_anomaly_ts','latest_anomaly_local',
    'max_value','global_P95','severity_ratio'
]].reset_index(drop=True)

# Return compact displays (and final_evidence is stored in kernel)
metrics_display, traces_display

```
Out[9]:
```
Compact evidence summary (MG02 and IG02)

MG02
- Metric (earliest relaxed segment)
  - kpi: OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_...
  - timestamp: 1614857400 → 2021-03-04 19:30:00+08
  - value = 0.300; global_P95 = 0.3000; global_P05 = 0.3000
  - segment_severity = 1.333

- Top matching trace evidence (sorted by severity_ratio)
  - trace.from_dockerA2.duration_mean — window 1614857820–1614858900 (2021-03-04 19:37–19:55), max=0.5036, global_P95=0.050976, severity≈9.88
  - dockerA2 → trace.to_MG02.duration_mean — same window, severity≈9.88
  - dockerA2 → trace.to_MG02.duration_p95 — same window, max=1.9838, global_P95=0.21752, severity≈9.12
  - trace.from_dockerA2.duration_p95 — same window, severity≈9.12
  - (Additional MG02 traces with severity 7.3, 3.38, etc., are available in the stored results.)

Interpretation for MG02:
- Metric shows an early relaxed anomaly at 19:30, while multiple very high-severity trace spikes (severity >> 1) occur shortly after (~19:37–19:55). The large trace severity ratios (≈9x) in MG02-related traces (dockerA2 interactions) indicate significant latency/spike on service paths involving MG02 — MG02 is a strong candidate for root cause. Full matching pairs are stored in kernel variables for detailed timeline inspection.

IG02
- Metric (earliest relaxed segment)
  - kpi: JVM-Memory_7778_JVM_Memory_HeapMemoryUsage
  - timestamp: 1614857880 → 2021-03-04 19:38:00+08
  - value = 12.502; global_P95 = 48.8105; global_P05 = 15.6783
  - segment_severity = 1.254

- Top matching trace evidence (sorted by severity_ratio)
  - trace.to_Tomcat03.duration_mean — window 1614857580–1614857640 (2021-03-04 19:33–19:34), max≈0.4004, global_P95≈0.2030, severity≈1.97
  - trace.from_IG02.duration_mean — same window, severity≈1.97
  - trace.to_Tomcat03.duration_p95 — window 1614857580–1614858660 (up to 19:51), max=2.266, global_P95≈1.15845, severity≈1.96
  - trace.from_IG02.duration_p95 — same window, severity≈1.96
  - trace.self.duration_mean — window 1614857400–1614857460, severity≈1.53

Interpretation for IG02:
- Metric shows a memory-related low anomaly at 19:38 (severity ~1.25). Trace anomalies involving IG02→Tomcat03 show moderate severity (~1.5–2.0) in windows that overlap the metric timeframe (some trace windows include 19:38). IG02 is a plausible secondary root cause candidate with moderate trace corroboration.

Overall conclusion and recommended next steps
- MG02: strongest evidence — high-severity trace spikes (dockerA2↔MG02) suggest MG02-related service latency/issues around 19:37–19:55. Prioritize MG02 and inspect the detailed spans and involved dockerA* services.
- IG02: moderate evidence — memory metric anomaly at 19:38 and moderate trace spikes to/from Tomcat03 around 19:33–19:51. Investigate IG02 memory usage and IG02→Tomcat03 traces for causality.
- Stored artifacts for drill-down: final_evidence (metrics + traces), component_overlap_matches, trace_anomalies, trace_anomaly_summary, relaxed_fault_segments. Use these to inspect exact span-level traces and per-minute metric samples.

The original code execution output of IPython Kernel is also provided below for reference:

(  cmdb_id                                           kpi_name   timestamp           timestamp_local   value  global_P95  global_P05  segment_severity
0    IG02         JVM-Memory_7778_JVM_Memory_HeapMemoryUsage  1614857880  2021-03-04 19:38:00+0800  12.502     48.8105     15.6783          1.254063
1    MG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...  1614857400  2021-03-04 19:30:00+0800   0.300      0.3000      0.3000          1.333333,   component trace_cmdb_id                         trace_name  anomaly_count  earliest_anomaly_ts    earliest_anomaly_local  latest_anomaly_ts      latest_anomaly_local  max_value  global_P95  severity_ratio
0      IG02          IG02    trace.to_Tomcat03.duration_mean              2           1614857580  2021-03-04 19:33:00+0800         1614857640  2021-03-04 19:34:00+0800   0.400439    0.203048        1.972143
1      IG02      Tomcat03      trace.from_IG02.duration_mean              2           1614857580  2021-03-04 19:33:00+0800         1614857640  2021-03-04 19:34:00+0800   0.400439    0.203048        1.972143
2      IG02          IG02     trace.to_Tomcat03.duration_p95              2           1614857580  2021-03-04 19:33:00+0800         1614858660  2021-03-04 19:51:00+0800   2.266000    1.158450        1.956062
3      IG02      Tomcat03       trace.from_IG02.duration_p95              2           1614857580  2021-03-04 19:33:00+0800         1614858660  2021-03-04 19:51:00+0800   2.266000    1.158450        1.956062
4      MG02          MG02  trace.from_dockerA2.duration_mean              2           1614857820  2021-03-04 19:37:00+0800         1614858900  2021-03-04 19:55:00+0800   0.503600    0.050976        9.879198
5      MG02      dockerA2        trace.to_MG02.duration_mean              2           1614857820  2021-03-04 19:37:00+0800         1614858900  2021-03-04 19:55:00+0800   0.503600    0.050976        9.879198
6      MG02      dockerA2         trace.to_MG02.duration_p95              2           1614857820  2021-03-04 19:37:00+0800         1614858900  2021-03-04 19:55:00+0800   1.983800    0.217520        9.120081
7      MG02          MG02   trace.from_dockerA2.duration_p95              2           1614857820  2021-03-04 19:37:00+0800         1614858900  2021-03-04 19:55:00+0800   1.983800    0.217520        9.120081)```
```